# Phase 2: Data Cleaning

This notebook performs data quality assessment and cleaning of the raw Books to Scrape dataset collected during Phase 1.

The cleaning process includes:

- inspection of data structure and data types;
- assessment of missing values;
- detection of duplicate records;
- validation of numeric variables;
- validation of UPC values;
- identification of suspicious categorical values;
- investigation of text-quality issues; and
- final validation of the processed dataset.

The raw dataset is preserved unchanged, and the cleaned dataset is saved separately in the `data/processed/` directory.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
RAW_PATH = Path("../data/raw/books_raw_data.csv")
PROCESSED_DIR = Path("../data/processed")

# Load raw dataset
df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")


Dataset loaded successfully.
Shape: (1000, 7)


In [2]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())


Column names:
['title', 'price', 'availability', 'rating', 'category', 'product_description', 'upc']

Data types:
title                      str
price                  float64
availability             int64
rating                   int64
category                   str
product_description        str
upc                        str
dtype: object

First 5 rows:


,title,price,availability,rating,category,product_description,upc
0,A Light in the Attic,51.77,22,3,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632
1,Tipping the Velvet,53.74,20,1,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a
2,Soumission,50.10,20,1,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760
3,Sharp Objects,47.82,20,4,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48
4,Sapiens: A Brief History of Humankind,54.23,20,5,History,From a renowned historian comes a groundbreaki...,4165285e1663650f


In [3]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

display(missing_summary)


,missing_count,missing_percentage
title,0,0.0
price,0,0.0
availability,0,0.0
rating,0,0.0
category,0,0.0
product_description,2,0.2
upc,0,0.0


In [4]:
print("Exact duplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate titles:")
print(df["title"].duplicated().sum())

print("\nDuplicate UPCs:")
print(df["upc"].duplicated().sum())

print("\nDuplicate Product Descriptions:")
print(df["product_description"].duplicated().sum())


Exact duplicate rows:
0

Duplicate titles:
1

Duplicate UPCs:
0

Duplicate Product Descriptions:
1


In [5]:
duplicate_titles = df[
    df["title"].duplicated(keep=False)
].sort_values("title")

display(
    duplicate_titles[
        ["title", "price", "availability", "rating", "category", "upc"]
    ]
)

duplicate_product_description = df[
    df["product_description"].duplicated(keep=False)
].sort_values("product_description")

display(
    duplicate_product_description[
        [
            "title",
            "price",
            "availability",
            "rating",
            "category",
            "product_description",
            "upc"
        ]
    ]
)


,title,price,availability,rating,category,upc
236,The Star-Touched Queen,46.02,14,5,Fantasy,1528279aec1f3dce
358,The Star-Touched Queen,32.30,12,5,Fantasy,4a7a25be293ad678


,title,price,availability,rating,category,product_description,upc
160,The Bridge to Consciousness: I'm Writing the B...,32.00,15,3,Default,NaN,efc3768127714ec3
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,1,1,Classics,NaN,cd2a2a70dd5d176d


### Duplicate Description Check

The duplicate-description count above can include missing values because pandas treats repeated `NaN` values as duplicated.

Therefore, the following check considers only non-missing descriptions when assessing true duplicate product descriptions.


In [6]:
non_missing_descriptions = df["product_description"].dropna()

true_duplicate_description_count = (
    non_missing_descriptions.duplicated().sum()
)

print(
    "True duplicate non-missing product descriptions:",
    true_duplicate_description_count
)


True duplicate non-missing product descriptions: 0


In [7]:
print("Price summary:")
display(df["price"].describe())

print("\nAvailability summary:")
display(df["availability"].describe())

print("\nRating summary:")
display(df["rating"].describe())


Price summary:


count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: price, dtype: float64


Availability summary:


count    1000.000000
mean        8.585000
std         5.654622
min         1.000000
25%         3.000000
50%         7.000000
75%        14.000000
max        22.000000
Name: availability, dtype: float64


Rating summary:


count    1000.000000
mean        2.923000
std         1.434967
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: rating, dtype: float64

In [8]:
print(
    "Invalid price values:",
    (df["price"] <= 0).sum()
)

print(
    "Invalid availability values:",
    (df["availability"] < 0).sum()
)

print(
    "Invalid ratings:",
    (~df["rating"].between(1, 5)).sum()
)


Invalid price values: 0
Invalid availability values: 0
Invalid ratings: 0


## UPC Validation

The UPC field is treated as a product identifier rather than a numerical measurement.

The expected structure is a 16-character hexadecimal string. The following checks verify missing values, uniqueness, length, and hexadecimal format.


In [9]:
upc_missing = df["upc"].isna().sum()
upc_unique = df["upc"].nunique(dropna=True)
upc_length_valid = df["upc"].dropna().astype(str).str.len().eq(16).sum()
upc_hex_valid = df["upc"].dropna().astype(str).str.fullmatch(
    r"[0-9a-fA-F]{16}"
).sum()

print("Missing UPCs:", upc_missing)
print("Unique UPCs:", upc_unique)
print("16-character UPCs:", upc_length_valid)
print("Valid hexadecimal UPCs:", upc_hex_valid)


Missing UPCs: 0
Unique UPCs: 1000
16-character UPCs: 1000
Valid hexadecimal UPCs: 1000


## Category Validation

The category variable is inspected for the number of unique categories and their frequencies.

No category is removed solely because its name appears unusual. In particular, `Default` and `Add a comment` are retained because they are valid categories in the source dataset.


In [10]:
print(f"Number of unique categories: {df['category'].nunique()}")

display(
    df["category"]
    .value_counts()
    .to_frame("count")
)


Number of unique categories: 50


,count
category,
Default,152
Nonfiction,110
Sequential Art,75
Add a comment,67
Fiction,65
Young Adult,54
Fantasy,48
Romance,35
Mystery,32


## Product Description Quality Investigation

The `product_description` field is investigated before the final cleaning decision.

The purpose is to identify:

- missing descriptions;
- character-encoding issues (mojibake); and
- repeated/truncated prefix artefacts.

The field will not be modified in the raw dataset. The investigation is used to determine whether the field should be retained in the processed modelling dataset.


In [11]:
print("Missing product descriptions:")
print(df["product_description"].isna().sum())

display(
    df[df["product_description"].isna()][
        [
            "title",
            "category",
            "upc"
        ]
    ]
)


Missing product descriptions:
2


,title,category,upc
160,The Bridge to Consciousness: I'm Writing the B...,Default,efc3768127714ec3
995,Alice in Wonderland (Alice's Adventures in Won...,Classics,cd2a2a70dd5d176d


In [12]:
def contains_mojibake(text):
    if pd.isna(text):
        return False

    pattern = r"(?:Ã.|Â.|â.|ð.|�)"

    return bool(
        pd.Series([text])
        .str.contains(pattern, regex=True)
        .iloc[0]
    )

print("Titles with possible mojibake:")
print(df["title"].apply(contains_mojibake).sum())

print("Product descriptions with possible mojibake:")
print(df["product_description"].apply(contains_mojibake).sum())


Titles with possible mojibake:
9
Product descriptions with possible mojibake:
562


In [13]:
def repair_mojibake(text):
    """
    Test repair for common UTF-8/Latin-1 mojibake.
    This function is used for investigation only.
    """
    if pd.isna(text) or not isinstance(text, str):
        return text

    try:
        repaired = text.encode("latin1").decode("utf-8")
        return repaired
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


encoding_test = df[
    df["product_description"].apply(contains_mojibake)
][
    ["title", "product_description"]
].head(10).copy()

encoding_test["repaired_description"] = (
    encoding_test["product_description"]
    .apply(repair_mojibake)
)

display(encoding_test)


,title,product_description,repaired_description
2,Soumission,"Dans une France assez proche de la nÃ´tre, un ...","Dans une France assez proche de la nÃ´tre, un ..."
3,Sharp Objects,"WICKED above her hipbone, GIRL across her hear...","WICKED above her hipbone, GIRL across her hear..."
4,Sapiens: A Brief History of Humankind,From a renowned historian comes a groundbreaki...,From a renowned historian comes a groundbreaki...
5,The Requiem Red,Patient Twenty-nine.A monster roams the halls ...,Patient Twenty-nine.A monster roams the halls ...
6,The Dirty Little Secrets of Getting Your Dream...,Drawing on his extensive experience evaluating...,Drawing on his extensive experience evaluating...
8,The Boys in the Boat: Nine Americans and Their...,For readers of Laura Hillenbrand's Seabiscuit ...,For readers of Laura Hillenbrand's Seabiscuit ...
9,The Black Maria,"Praise for Aracelis Girmay:""[Girmay's] every l...","Praise for Aracelis Girmay:""[Girmay's] every l..."
12,Set Me Free,Aaron Ledbetterâs future had been planned ou...,Aaron Ledbetter’s future had been planned out ...
15,Our Band Could Be Your Life: Scenes from the A...,This is the never-before-told story of the mus...,This is the never-before-told story of the mus...
16,Olio,"Part fact, part fiction, Tyehimba Jess's much ...","Part fact, part fiction, Tyehimba Jess's much ..."


In [14]:
def find_repeated_prefix(text, min_prefix_length=50):
    """
    Detect whether the beginning of a description
    appears again later in the same string.
    """
    if pd.isna(text) or not isinstance(text, str):
        return None

    prefix = text[:min_prefix_length]

    second_position = text.find(
        prefix,
        min_prefix_length
    )

    if second_position == -1:
        return None

    return second_position


df["repeated_prefix_position"] = (
    df["product_description"]
    .apply(find_repeated_prefix)
)

repeated_prefix_count = (
    df["repeated_prefix_position"]
    .notna()
    .sum()
)

print(
    "Descriptions with repeated prefix:",
    repeated_prefix_count
)


Descriptions with repeated prefix: 925


In [15]:
repeated_prefix_examples = df[
    df["repeated_prefix_position"].notna()
][
    [
        "title",
        "product_description",
        "repeated_prefix_position"
    ]
]

display(
    repeated_prefix_examples.head(10)
)


,title,product_description,repeated_prefix_position
0,A Light in the Attic,It's hard to imagine a world without A Light i...,376.0
1,Tipping the Velvet,"""Erotic and absorbing...Written with starling ...",375.0
2,Soumission,"Dans une France assez proche de la nÃ´tre, un ...",392.0
3,Sharp Objects,"WICKED above her hipbone, GIRL across her hear...",379.0
4,Sapiens: A Brief History of Humankind,From a renowned historian comes a groundbreaki...,388.0
5,The Requiem Red,Patient Twenty-nine.A monster roams the halls ...,378.0
6,The Dirty Little Secrets of Getting Your Dream...,Drawing on his extensive experience evaluating...,377.0
7,The Coming Woman: A Novel Based on the Life of...,"""If you have a heart, if you have a soul, Kare...",375.0
8,The Boys in the Boat: Nine Americans and Their...,For readers of Laura Hillenbrand's Seabiscuit ...,380.0
9,The Black Maria,"Praise for Aracelis Girmay:""[Girmay's] every l...",382.0


## Final Cleaning Decision: Product Description

The `product_description` variable was investigated as part of the data-quality assessment.

The analysis identified several issues with this variable:

- it contains long free-text strings;
- some observations contain character-encoding issues (mojibake);
- some descriptions contain repeated or truncated prefix artefacts; and
- there are missing product descriptions.

Although these issues could potentially be repaired using text-processing techniques, repairing the field would add unnecessary complexity to the current structured-price modelling task.

Therefore, `product_description` is excluded from the processed modelling dataset.

The original `product_description` field remains unchanged in the raw dataset for traceability.

No product records are removed because of these description issues.


In [16]:
# Remove investigation-only helper column before creating the processed dataset
df_processed = df.drop(
    columns=["repeated_prefix_position"]
).copy()

# Remove product_description from the modelling dataset
df_processed = df_processed.drop(
    columns=["product_description"]
).copy()

print("Processed dataset shape:", df_processed.shape)

print("\nRemaining columns:")
print(df_processed.columns.tolist())


Processed dataset shape: (1000, 6)

Remaining columns:
['title', 'price', 'availability', 'rating', 'category', 'upc']


## Final Dataset Validation

The processed dataset is validated after the cleaning decisions have been applied.

The checks confirm:

- the expected number of observations;
- no missing values in the retained variables;
- no exact duplicate rows;
- no duplicate UPCs;
- valid price values;
- valid availability values; and
- valid rating values.


In [17]:
print("Dataset shape:", df_processed.shape)

print("\nMissing values:")
display(df_processed.isna().sum())

print("\nDuplicate rows:", df_processed.duplicated().sum())
print("Duplicate UPCs:", df_processed["upc"].duplicated().sum())

print("\nPrice range:")
print(
    df_processed["price"].min(),
    "to",
    df_processed["price"].max()
)

print("\nAvailability range:")
print(
    df_processed["availability"].min(),
    "to",
    df_processed["availability"].max()
)

print("\nRating range:")
print(
    df_processed["rating"].min(),
    "to",
    df_processed["rating"].max()
)


Dataset shape: (1000, 6)

Missing values:


title           0
price           0
availability    0
rating          0
category        0
upc             0
dtype: int64


Duplicate rows: 0
Duplicate UPCs: 0

Price range:
10.0 to 59.99

Availability range:
1 to 22

Rating range:
1 to 5


In [18]:
print("Final category count:")
print(df_processed["category"].nunique())

display(
    df_processed["category"]
    .value_counts()
    .head(15)
)


Final category count:
50


category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               19
History                18
Name: count, dtype: int64

## Save Processed Dataset

The cleaned dataset is saved separately from the raw dataset.

The raw CSV is not overwritten.


In [19]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_PATH = PROCESSED_DIR / "books_processed_data.csv"

df_processed.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8"
)

print(
    f"Processed dataset saved to: {PROCESSED_PATH}"
)


Processed dataset saved to: ..\data\processed\books_processed_data.csv


## Phase 2 Summary

The Phase 2 data cleaning process has been completed.

The final processed dataset retains the following variables:

- `title`
- `price`
- `availability`
- `rating`
- `category`
- `upc`

The `product_description` variable was removed because it contains long free-text data and identified text-quality issues, including encoding and source-format artefacts, as well as missing values.

No observations were removed during the cleaning process.

The duplicated title was retained because the corresponding records have different product information and different UPC values, so they do not represent exact duplicate products.

All source categories were retained, including `Default` and `Add a comment`.

The final validation confirms that the retained variables contain no missing values, no duplicate rows, no duplicate UPCs, and valid numerical ranges.

The processed dataset is now saved in `data/processed/books_processed_data.csv` and is ready for the next stage of the project.
